# 05 · Análisis exploratorio: ensamble y segundas diferencias

**Etapa 8 del plan.** Responde dos preguntas que el `04` dejó abiertas:
1. **Ensamble:** ¿la ventaja de la historia se sostiene al promediar las 10 semillas?
2. **Segundas diferencias:** ¿agregan algo? El temario las nombra para este tema y no se probaron.

**Todo lo de este notebook es exploratorio.** 201909 y 201910 ya se miraron, así que nada de acá demuestra una hipótesis ni cambia el resultado de la regla pre-registrada. Las comparaciones son solo las fijadas en la ficha, antes de ver estos resultados.

**No escribe en `exp/03/` ni en los `resultados/03_*` y `04_*`:** solo los lee.

Ficha: `fichas/etapa_8_exploratorio.md` (en el laboratorio).

## 1. Parámetros

- **Retomar:** si la corrida se corta, se vuelve a correr entera. Las semillas ya hechas no se repiten.
- **Corrida de prueba:** `N_SEMILLAS_D2 = 2`, en un directorio aparte.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import time
import warnings

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP_03 = Path('../exp/03')                          # solo se lee
EXP = Path('../exp/05')
FIGURAS = RESULTADOS / 'figuras'

CONTROL = 'A'
CONFIGS_03 = ['A', 'L3', 'L4', 'T3', 'T4', 'LT3', 'LT4', 'A-ranking', 'LT4-ranking', 'A-consigna']
COMPARAR_ENSAMBLE = ['L3', 'L4', 'T3', 'T4', 'LT3', 'LT4', 'A-consigna']   # cada una contra A
COMPARAR_D2 = {'D2': 'A', 'LT3D2': 'LT3'}                                  # configuración nueva: contra cuál

MESES_TRAIN = [201905, 201906]
MES_VALID = 201908
MESES_TEST = [201909, 201910]
MESES_EVAL = [MES_VALID] + MESES_TEST

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 10                                     # las del 03, para el ensamble
N_SEMILLAS_D2 = 10                                  # modelos nuevos por configuración
N_BOOTSTRAP = 2_000
NIVEL_IC = 0.95
BANDA = (3_000, 6_000)                              # ambos incluidos
POSITIVOS = ['BAJA+1', 'BAJA+2']
N_HILOS = 8

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
VENTANA_SUAVIZADO = 501
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

PRED_03 = EXP_03 / 'pred'
PRED = EXP / 'pred'
for d in (PRED, FIGURAS):
    d.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14

ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116, 201905: 126_763,
    201906: 128_370, 201907: 129_853, 201908: 131_775, 201909: 133_419, 201910: 135_633,
}
ESPERADO_EVAL = {201908: (131_775, 543), 201909: (133_419, 615), 201910: (135_633, 632)}   # filas, BAJA+2
ESPERADO_N_VARIABLES = {'A': 760, 'D2': 889, 'LT3D2': 1_697}
ESPERADO_ROTOS = 23


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


SEMILLAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()[:N_SEMILLAS]
assert SEMILLAS == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
SEMILLAS_D2 = SEMILLAS[:N_SEMILLAS_D2]


def huella_archivos():
    """Control 8: huella de las salidas del 03 y del 04, para comprobar que no cambian."""
    return {p.name: hashlib.blake2b(p.read_bytes(), digest_size=16).hexdigest()
            for p in sorted(RESULTADOS.glob('0[34]_*'))}


HUELLA_INICIAL = huella_archivos()
T_INICIO = time.time()
print('Salidas del 03 y del 04 registradas:', len(HUELLA_INICIAL))

## 2. Funciones y carga

Las mismas definiciones del `03` y del `04`: curva de ganancia, N en la meseta de la curva de 201908, banda, Wilcoxon exacto e IC con t de Student.

In [ ]:
def curva(prob, es_b2):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:LARGO_CURVA].astype(np.float64)


def elegir_n(curva_media):
    suave = pd.Series(curva_media).rolling(VENTANA_SUAVIZADO, center=True, min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    return int(np.nanargmax(suave)) + 1


def banda(cv):
    return cv[BANDA[0] - 1:BANDA[1]].mean()


def wilcoxon_exacto(d):
    d = np.round(np.asarray(d, dtype=np.float64), 6)
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 1.0
    rangos = stats.rankdata(np.abs(d))
    w = float((np.sign(d) * rangos).sum())
    signos = 1 - 2 * ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1)
    return float(np.mean(np.abs(signos @ rangos) >= abs(w) - 1e-9))


def ic(x):
    x = np.asarray(x, dtype=np.float64)
    media = x.mean()
    medio = stats.t.ppf((1 + NIVEL_IC) / 2, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return media, media - medio, media + medio


assert abs(wilcoxon_exacto(np.arange(1, 11)) - 2 / 1024) < 1e-12
assert abs(wilcoxon_exacto([1, 1, 1, 1, 1, 1, 1, 1, 1, -1]) - 22 / 1024) < 1e-12

tabla_n = pd.read_csv(RESULTADOS / '03_N.csv').set_index('config')
N_03 = tabla_n['N'].astype(int).to_dict()
ganancias_04 = pd.read_csv(RESULTADOS / '04_ganancias.csv')
validacion = pd.read_csv(RESULTADOS / '03_validacion.csv').set_index(['config', 'semilla'])
HIPER = json.loads((RESULTADOS / '03_hiperparametros.json').read_text())

claves_npz = np.load(EXP_03 / 'claves_eval.npz')
claves = pd.DataFrame({'numero_de_cliente': claves_npz['numero_de_cliente'], 'foto_mes': claves_npz['foto_mes']})
PROB = {c: np.stack([np.load(PRED_03 / c / f'{s}.npy') for s in SEMILLAS]) for c in CONFIGS_03}

con = duckdb.connect()
clase = con.execute(f"""
    select numero_de_cliente, foto_mes, clase_ternaria
    from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes in ({', '.join(str(m) for m in MESES_EVAL)})
""").df()
con.close()
eval_ = claves.merge(clase, on=['numero_de_cliente', 'foto_mes'], how='left', validate='one_to_one')
assert len(eval_) == len(claves) and eval_['clase_ternaria'].notna().all()
MES = eval_['foto_mes'].to_numpy()
ES_B2 = (eval_['clase_ternaria'] == 'BAJA+2').to_numpy()
es_mes = {m: MES == m for m in MESES_EVAL}
for mes, (filas, b2) in ESPERADO_EVAL.items():            # control 1
    assert (es_mes[mes].sum(), ES_B2[es_mes[mes]].sum()) == (filas, b2), mes
print(f'{len(PROB)} configuraciones × {N_SEMILLAS} semillas cargadas')

## 3. Parte 1: ensamble

- **Ensamble:** promedio de las probabilidades de las 10 semillas de cada configuración (decisión 1).
- **N:** se elige de nuevo sobre la curva del ensamble en 201908, con el método del pre-registro (decisión 2). Se aplica sin cambios a 201909 y 201910.

**Control:** con una sola semilla, el ensamble tiene que dar la ganancia de `04_ganancias.csv`.

In [ ]:
def ensamblar(probs):
    return np.asarray(probs, dtype=np.float64).mean(axis=0)


for m in MESES_EVAL:                                      # control 2
    for c in ('A', 'LT4'):
        uno = ensamblar(PROB[c][:1])
        g = curva(uno[es_mes[m]], ES_B2[es_mes[m]])[N_03[c] - 1] / 1e6
        esperado = ganancias_04.query('config == @c and foto_mes == @m and semilla == @SEMILLAS[0]')['ganancia_M'].item()
        assert abs(g - esperado) < 6e-5, (c, m, g, esperado)

ENSAMBLE = {c: ensamblar(PROB[c]) for c in CONFIGS_03}
CURVA_E = {(c, m): curva(ENSAMBLE[c][es_mes[m]], ES_B2[es_mes[m]]) for c in CONFIGS_03 for m in MESES_EVAL}
N_E = {c: elegir_n(CURVA_E[(c, MES_VALID)]) for c in CONFIGS_03}

media_semillas = ganancias_04.groupby(['config', 'foto_mes'])['ganancia_M'].mean()
filas = []
for c in CONFIGS_03:
    for m in MESES_EVAL:
        cv = CURVA_E[(c, m)]
        filas.append({'config': c, 'foto_mes': m, 'N_ensamble': N_E[c], 'N_03': N_03[c],
                      'ganancia_ensamble_M': round(cv[N_E[c] - 1] / 1e6, 4),
                      'ganancia_ensamble_N03_M': round(cv[N_03[c] - 1] / 1e6, 4),
                      'ganancia_banda_M': round(banda(cv) / 1e6, 4),
                      'ganancia_media_semillas_M': round(media_semillas[(c, m)], 4),
                      'auc': round(roc_auc_score(ES_B2[es_mes[m]], ENSAMBLE[c][es_mes[m]]), 6)})
ensamble = pd.DataFrame(filas)
ensamble.to_csv(RESULTADOS / '05_ensamble.csv', index=False)
print('EXPLORATORIO')
print(ensamble[ensamble['foto_mes'] != MES_VALID].to_string(index=False))

### Variación del ensamble: bootstrap pareado de clientes

Hay un solo ensamble por configuración, así que no hay semillas para comparar (decisión 3). Se remuestrean los clientes del mes, con reposición, **los mismos para A y para la variante**:
- los clientes contactados quedan fijos;
- cada cliente aporta la diferencia entre lo que deja en una configuración y en la otra;
- el intervalo son los percentiles 2,5 y 97,5 de 2.000 remuestreos.

Mide algo que las semillas no medían: cuánto depende el resultado de los clientes de ese mes.

**Control:** comparar A contra A da 0 en todos los remuestreos, y dos corridas con la misma semilla dan lo mismo.

In [ ]:
def aporte(prob, es_b2, n):
    """Lo que deja cada cliente del mes: 0 si no se lo contacta."""
    contactado = np.zeros(len(prob), dtype=bool)
    contactado[np.argsort(-prob, kind='stable')[:n]] = True
    return np.where(contactado, np.where(es_b2, GANANCIA_ACIERTO, -COSTO_ESTIMULO), 0).astype(np.float64)


def remuestrear(D, n_veces, semilla):
    """Suma de cada columna de D en n_veces remuestreos de las filas, con reposición."""
    rng = np.random.default_rng(semilla)
    n = D.shape[0]
    con_valor = np.flatnonzero(D.any(axis=1))
    salida = np.empty((n_veces, D.shape[1]))
    for b in range(n_veces):
        veces = np.bincount(rng.integers(0, n, size=n), minlength=n)
        salida[b] = veces[con_valor] @ D[con_valor]
    return salida


filas = []
for m in MESES_TEST:
    b2 = ES_B2[es_mes[m]]
    base_a = aporte(ENSAMBLE[CONTROL][es_mes[m]], b2, N_E[CONTROL])
    assert base_a.sum() == CURVA_E[(CONTROL, m)][N_E[CONTROL] - 1]
    D = np.column_stack([aporte(ENSAMBLE[c][es_mes[m]], b2, N_E[c]) - base_a for c in COMPARAR_ENSAMBLE])

    assert (remuestrear((base_a - base_a)[:, None], 20, SEMILLA_PRIMIGENIA) == 0).all()     # control 3
    np.testing.assert_array_equal(remuestrear(D, 20, SEMILLA_PRIMIGENIA), remuestrear(D, 20, SEMILLA_PRIMIGENIA))

    sumas = remuestrear(D, N_BOOTSTRAP, SEMILLA_PRIMIGENIA + m) / 1e6
    for j, c in enumerate(COMPARAR_ENSAMBLE):
        filas.append({'config': c, 'contra': CONTROL, 'foto_mes': m, 'N': N_E[c], 'N_contra': N_E[CONTROL],
                      'd_M': D[:, j].sum() / 1e6,
                      'ic95_inf_M': np.percentile(sumas[:, j], 2.5), 'ic95_sup_M': np.percentile(sumas[:, j], 97.5),
                      'remuestreos_a_favor_pct': (sumas[:, j] > 0).mean() * 100,
                      'clientes_distintos': int((D[:, j] != 0).sum()),
                      'd_banda_M': (banda(CURVA_E[(c, m)]) - banda(CURVA_E[(CONTROL, m)])) / 1e6})
ensamble_vs_a = pd.DataFrame(filas)
ensamble_vs_a.round(4).to_csv(RESULTADOS / '05_ensamble_vs_A.csv', index=False)
print('EXPLORATORIO')
print(ensamble_vs_a.round(3).to_string(index=False))

## 4. Parte 2: segundas diferencias

**Definición:** `d2 = y_t − 2·y_(t−1) + y_(t−2)`, sobre los 129 montos y conteos. Solo usa meses que A ya mira: **no agrega historia, agrega otra forma de leerla**. Si falta alguno de los tres meses, queda NA.

Todo lo demás es igual al `03`: ventana, meses rotos, hiperparámetros, árboles y semillas.

In [ ]:
t0 = time.time()
con = duckdb.connect()
base = con.execute(f"""
    select * from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {max(MESES_TEST)}
    order by foto_mes, numero_de_cliente
""").df()
con.close()
assert base['foto_mes'].value_counts().sort_index().to_dict() == ESPERADO_FILAS_POR_MES

CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]
BANDERAS = ['active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
            'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones']
ESTADOS = ['tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency']
FECHAS = [f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')]
EDADES = ['cliente_edad', 'cliente_antiguedad']
MONTOS = [v for v in VARIABLES if v not in BANDERAS + ESTADOS + FECHAS + EDADES]
assert len(VARIABLES) == 152 and len(MONTOS) == 129

rotos = pd.read_csv(RESULTADOS / '01_meses_rotos.csv')
assert len(rotos) == ESPERADO_ROTOS
for v, mes in rotos[['variable', 'foto_mes']].itertuples(index=False):
    base.loc[base['foto_mes'] == mes, v] = np.nan

FOTO_MES = base['foto_mes'].to_numpy()
PERIODO = (FOTO_MES // 100 * 12 + FOTO_MES % 100).astype(np.int64)
CLAVE = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + PERIODO
INDICE = pd.Index(CLAVE)
assert INDICE.is_unique
X_MES = np.asfortranarray(base[VARIABLES].to_numpy(np.float32))
CLASE = base['clase_ternaria'].to_numpy()
del base
gc.collect()
print(f'Dataset cargado en {time.time() - t0:.0f} s')

VARIANTES = {
    'A':     {'lags': [],  'tendencia': None, 'd2': False},
    'D2':    {'lags': [],  'tendencia': None, 'd2': True},
    'LT3D2': {'lags': [3], 'tendencia': 3,    'd2': True},
}


def especificacion(variante):
    ex = VARIANTES[variante]
    e = [('original', v, 0) for v in VARIABLES]
    e += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
    for k in ex['lags']:
        e += [(op, v, k) for op in ('lag', 'delta') for v in MONTOS + BANDERAS]
        e += [(op, v, k) for op in ('lag', 'cambio') for v in ESTADOS]
    if ex['tendencia']:
        h = ex['tendencia']
        e += [(op, v, h) for op in ('pend', 'prom', 'min', 'max') for v in MONTOS]
        e += [('perm', v, h) for v in BANDERAS]
    if ex['d2']:
        e += [('d2', v, 2) for v in MONTOS]
    return e


def nombre(op, v, k):
    return v if op == 'original' else f'{v}_{op}{k}'


def construir(variante):
    """El cálculo del 03, con las segundas diferencias al final."""
    ex = VARIANTES[variante]
    h = ex['tendencia']
    h_max = max([2] + ex['lags'] + [h or 0])
    filas = np.flatnonzero(np.isin(FOTO_MES, MESES_TRAIN + MESES_EVAL))
    mes_fila = FOTO_MES[filas]
    n_train = int(np.isin(mes_fila, MESES_TRAIN).sum())
    assert np.isin(mes_fila[:n_train], MESES_TRAIN).all() and np.isin(mes_fila[n_train:], MESES_EVAL).all()
    pos = {k: INDICE.get_indexer(CLAVE[filas] - k) for k in range(h_max + 1)}

    espec = especificacion(variante)
    ubic = {e: i for i, e in enumerate(espec)}
    F = np.empty((len(filas), len(espec)), dtype=np.float32)
    t = -np.arange((h or 0) + 1, dtype=np.float64)[:, None]
    with np.errstate(invalid='ignore', divide='ignore'):
        for j, v in enumerate(VARIABLES):
            F[:, ubic[('original', v, 0)]] = X_MES[filas, j]
            hk = {}
            for k in range(h_max + 1):
                col = X_MES[pos[k], j]
                col[pos[k] < 0] = np.nan
                hk[k] = col
            for k in range(1, h_max + 1):
                if ('lag', v, k) in ubic:
                    F[:, ubic[('lag', v, k)]] = hk[k]
                if ('delta', v, k) in ubic:
                    F[:, ubic[('delta', v, k)]] = hk[0] - hk[k]
                if ('cambio', v, k) in ubic:
                    a, b = hk[0], hk[k]
                    F[:, ubic[('cambio', v, k)]] = np.where(np.isnan(a) | np.isnan(b), np.nan, (a != b).astype(np.float32))
            if ('d2', v, 2) in ubic:
                F[:, ubic[('d2', v, 2)]] = hk[0] - 2 * hk[1] + hk[2]
            if h and (('prom', v, h) in ubic or ('perm', v, h) in ubic):
                S = np.stack([hk[k] for k in range(h + 1)]).astype(np.float64)
                valido = ~np.isnan(S)
                cuantos = valido.sum(axis=0)
                S0 = np.where(valido, S, 0.0)
                if ('perm', v, h) in ubic:
                    F[:, ubic[('perm', v, h)]] = np.where(cuantos > 0, S0.sum(axis=0), np.nan)
                else:
                    prom = S0.sum(axis=0) / cuantos
                    dt = np.where(valido, t - (valido * t).sum(axis=0) / cuantos, 0.0)
                    pend = (dt * (S0 - prom)).sum(axis=0) / (dt ** 2).sum(axis=0)
                    pend[cuantos < 2] = np.nan
                    F[:, ubic[('pend', v, h)]] = pend
                    F[:, ubic[('prom', v, h)]] = prom
                    F[:, ubic[('min', v, h)]] = np.where(cuantos > 0, np.where(valido, S, np.inf).min(axis=0), np.nan)
                    F[:, ubic[('max', v, h)]] = np.where(cuantos > 0, np.where(valido, S, -np.inf).max(axis=0), np.nan)
    return {'F': F, 'columnas': [nombre(*e) for e in espec], 'filas': filas, 'mes_fila': mes_fila, 'n_train': n_train}


def huella_control(T):
    hsh = hashlib.blake2b(digest_size=16)
    hsh.update(T['filas'].tobytes())
    for j in range(ESPERADO_N_VARIABLES['A']):
        hsh.update(T['F'][:, j].tobytes())
    return hsh.hexdigest()


HUELLA_03 = json.loads((EXP_03 / 'hashes_control.json').read_text())['A']


def controlar(T, variante):
    F, col, mes_fila = T['F'], {c: i for i, c in enumerate(T['columnas'])}, T['mes_fila']
    assert F.shape[1] == ESPERADO_N_VARIABLES[variante], F.shape          # control 4
    assert huella_control(T) == HUELLA_03, 'Las columnas de A no son las del 03'   # control 5
    eval_t = slice(T['n_train'], None)
    assert (CLAVE[T['filas'][eval_t]] // 100_000 == claves['numero_de_cliente'].to_numpy()).all()
    assert (T['mes_fila'][eval_t] == claves['foto_mes'].to_numpy()).all()
    if not VARIANTES[variante]['d2']:
        return
    v = 'ctrx_quarter'                                                    # control 7: d2 a mano
    j = VARIABLES.index(v)
    valores = pd.Series(X_MES[:, j].astype(np.float64), index=INDICE)
    rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
    muestra = rng.choice(len(T['filas']), size=2_000, replace=False)
    y = np.stack([valores.reindex(CLAVE[T['filas'][muestra]] - k).to_numpy() for k in range(3)])
    completos = np.flatnonzero(~np.isnan(y).any(axis=0))
    for i in rng.choice(completos, size=3, replace=False):
        np.testing.assert_allclose(F[muestra[i], col[f'{v}_d22']], y[0, i] - 2 * y[1, i] + y[2, i], rtol=1e-5)
    incompletos = np.flatnonzero(np.isnan(y).any(axis=0))
    assert np.isnan(F[muestra[incompletos], col[f'{v}_d22']]).all()
    periodo_fila = mes_fila // 100 * 12 + mes_fila % 100                  # los meses rotos no vuelven por d2
    for v, mes in rotos[['variable', 'foto_mes']].itertuples(index=False):
        if v in MONTOS:
            p = (mes // 100) * 12 + mes % 100
            for k in range(3):
                en_destino = periodo_fila == p + k
                if en_destino.any():
                    assert np.isnan(F[en_destino, col[f'{v}_d22']]).all(), (v, mes, k)

### Modelos

**Control:** antes de entrenar lo nuevo, se reentrena `A` con la primera semilla. Tiene que dar lo mismo que el `03`.

In [ ]:
FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}
MEJORES = {**FIJOS, **HIPER['params']}
ARBOLES = HIPER['arboles']


def datos(T):
    n = T['n_train']
    y_train = np.isin(CLASE[T['filas'][:n]], POSITIVOS).astype(np.int8)
    return lgb.Dataset(T['F'][:n], label=y_train, feature_name=T['columnas'],
                       params={'max_bin': 31, 'feature_pre_filter': False}, free_raw_data=False)


T = construir('A')                                         # control 6
controlar(T, 'A')
modelo = lgb.train({**MEJORES, 'seed': SEMILLAS[0]}, datos(T), num_boost_round=ARBOLES)
prob = modelo.predict(T['F'][T['n_train']:], num_threads=N_HILOS).astype(np.float32)
auc = roc_auc_score(ES_B2[es_mes[MES_VALID]], prob[es_mes[MES_VALID]])
esperada = validacion.loc[('A', SEMILLAS[0]), 'auc_201908']
print(f'Reproducción de A: AUC 201908 {auc:.6f} · la del 03: {esperada:.6f} ({HIPER["maquina"]})')
if HIPER['maquina'] == platform.node():
    assert abs(auc - esperada) < 6e-7
    assert np.abs(prob - PROB['A'][0]).max() < 1e-6
del T, modelo, prob
gc.collect()

segundos = {}
for config in COMPARAR_D2:
    (PRED / config).mkdir(exist_ok=True)
    pendientes = [s for s in SEMILLAS_D2 if not (PRED / config / f'{s}.npy').exists()]
    if not pendientes:
        print(f'{config}: las {len(SEMILLAS_D2)} semillas ya estaban')
        continue
    t0 = time.time()
    T = construir(config)
    controlar(T, config)
    dtrain = datos(T)
    print(f'{config}: {T["F"].shape[1]:,} variables · tabla y controles en {time.time() - t0:.0f} s')
    for semilla in pendientes:
        t0 = time.time()
        modelo = lgb.train({**MEJORES, 'seed': semilla}, dtrain, num_boost_round=ARBOLES)
        prob = modelo.predict(T['F'][T['n_train']:], num_threads=N_HILOS).astype(np.float32)
        np.save(PRED / config / f'{semilla}.npy', prob)
        segundos.setdefault(config, []).append(time.time() - t0)
        print(f'{config} · semilla {semilla}: {time.time() - t0:.0f} s')
    del T, dtrain, modelo, prob
    gc.collect()

### Comparación

Para cada configuración nueva: N con el método del pre-registro y la diferencia por semilla contra su par, en 201909 y 201910. El p-valor es **descriptivo**, sin corrección.

**LT3 se eligió mirando el test**, porque fue la variante de mayor diferencia media. Por eso la comparación contra LT3 es la más débil del notebook.

In [ ]:
faltan = [(c, s) for c in COMPARAR_D2 for s in SEMILLAS_D2 if not (PRED / c / f'{s}.npy').exists()]
assert not faltan, faltan                                  # control 9
PROB_D2 = {c: np.stack([np.load(PRED / c / f'{s}.npy') for s in SEMILLAS_D2]) for c in COMPARAR_D2}


def por_semilla(probs, m):
    return np.stack([curva(p[es_mes[m]], ES_B2[es_mes[m]]) for p in probs])


pares = {**{c: PROB_D2[c] for c in COMPARAR_D2}, **{c: PROB[c][:N_SEMILLAS_D2] for c in COMPARAR_D2.values()}}
N_D2 = {c: elegir_n(por_semilla(p, MES_VALID).mean(axis=0)) for c, p in pares.items()}
if N_SEMILLAS_D2 == N_SEMILLAS:
    assert all(N_D2[c] == N_03[c] for c in COMPARAR_D2.values())   # los pares usan el N del 03

filas = []
G, B = {}, {}
for c, p in pares.items():
    for m in MESES_EVAL:
        cv = por_semilla(p, m)
        G[(c, m)] = cv[:, N_D2[c] - 1] / 1e6
        B[(c, m)] = cv[:, BANDA[0] - 1:BANDA[1]].mean(axis=1) / 1e6
        for i, s in enumerate(SEMILLAS_D2):
            filas.append({'config': c, 'semilla': s, 'foto_mes': m, 'N': N_D2[c], 'ganancia_M': round(G[(c, m)][i], 4),
                          'ganancia_banda_M': round(B[(c, m)][i], 4),
                          'auc': round(roc_auc_score(ES_B2[es_mes[m]], p[i][es_mes[m]]), 6)})
d2 = pd.DataFrame(filas)
d2.to_csv(RESULTADOS / '05_d2.csv', index=False)

filas = []
for c, contra in COMPARAR_D2.items():
    for m in MESES_TEST:
        d = (G[(c, m)] - G[(contra, m)]).round(6)
        media, inf, sup = ic(d)
        filas.append({'config': c, 'contra': contra, 'foto_mes': m, 'N': N_D2[c], 'N_contra': N_D2[contra],
                      'ganancia_media_M': G[(c, m)].mean(), 'ganancia_media_contra_M': G[(contra, m)].mean(),
                      'd_media_M': media, 'ic95_inf_M': inf, 'ic95_sup_M': sup,
                      'semillas_a_favor': int((d > 0).sum()), 'p_descriptivo': wilcoxon_exacto(d),
                      'd_banda_M': (B[(c, m)] - B[(contra, m)]).mean()})
d2_comparacion = pd.DataFrame(filas)
d2_comparacion.round(4).to_csv(RESULTADOS / '05_d2_comparacion.csv', index=False)
print('EXPLORATORIO')
print(d2_comparacion.round(3).to_string(index=False))

## 5. Figuras

Tamaño 1920 × 1080 (16:9). Todas llevan el rótulo de exploratorias.

In [ ]:
TINTA, TINTA_2, GRIS, GRILLA = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
AZUL, NARANJA = '#2a78d6', '#eb6834'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA_2, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'text.color': TINTA,
                     'axes.grid': True, 'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 13, 'axes.titlesize': 15})
TAMANO, DPI = (12.8, 7.2), 150


def coma(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


# Figura 1: ensamble contra la media de las semillas
orden = ['A', 'L3', 'L4', 'T3', 'T4', 'LT3', 'LT4', 'A-consigna']
fig, ejes = plt.subplots(1, 2, figsize=TAMANO)
for ax, m in zip(ejes, MESES_TEST):
    t = ensamble[ensamble['foto_mes'] == m].set_index('config').loc[orden]
    y = np.arange(len(orden))[::-1]
    ax.hlines(y, t['ganancia_media_semillas_M'], t['ganancia_ensamble_M'], color=GRIS, lw=1.5)
    ax.scatter(t['ganancia_media_semillas_M'], y, s=70, color=GRIS, zorder=3, label='media de las semillas', edgecolor='#fcfcfb')
    ax.scatter(t['ganancia_ensamble_M'], y, s=70, color=AZUL, zorder=3, label='ensamble', edgecolor='#fcfcfb')
    ax.set_yticks(y, orden)
    ax.set_title(str(m), loc='left')
    ax.set_xlabel('Ganancia (M$)')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: coma(v, 0)))
ejes[1].legend(frameon=False, loc='lower right')
fig.suptitle('Exploratorio: ganancia del ensamble contra la media de las semillas', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '05_1_ensamble_vs_semillas.png', dpi=DPI)
plt.show()


def intervalos(ax, etiquetas, media, inf, sup):
    y = np.arange(len(etiquetas))[::-1]
    ax.axvline(0, color=TINTA_2, lw=1.2)
    ax.hlines(y, inf, sup, color=AZUL, lw=2)
    ax.scatter(media, y, s=70, color=AZUL, zorder=3, edgecolor='#fcfcfb')
    ax.set_yticks(y, etiquetas)
    ax.set_ylim(-0.7, len(etiquetas) - 0.3)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: coma(v, 0)))


# Figura 2: diferencia del ensamble contra A
fig, ejes = plt.subplots(1, 2, figsize=TAMANO, sharex=True)
for ax, m in zip(ejes, MESES_TEST):
    t = ensamble_vs_a[ensamble_vs_a['foto_mes'] == m].set_index('config').loc[COMPARAR_ENSAMBLE]
    intervalos(ax, COMPARAR_ENSAMBLE, t['d_M'], t['ic95_inf_M'], t['ic95_sup_M'])
    ax.set_title(str(m), loc='left')
    ax.set_xlabel('Diferencia contra A (M$), IC 95 %')
fig.suptitle('Exploratorio: diferencia del ensamble contra A (bootstrap de clientes)', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '05_2_ensamble_vs_A.png', dpi=DPI)
plt.show()

# Figura 3: segundas diferencias
fig, ejes = plt.subplots(1, 2, figsize=TAMANO, sharex=True)
etiquetas = [f'{c} − {contra}' for c, contra in COMPARAR_D2.items()]
for ax, m in zip(ejes, MESES_TEST):
    t = d2_comparacion[d2_comparacion['foto_mes'] == m].set_index('config').loc[list(COMPARAR_D2)]
    intervalos(ax, etiquetas, t['d_media_M'], t['ic95_inf_M'], t['ic95_sup_M'])
    ax.set_title(str(m), loc='left')
    ax.set_xlabel('Diferencia media por semilla (M$), IC 95 %')
fig.suptitle('Exploratorio: qué agregan las segundas diferencias', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '05_3_segundas_diferencias.png', dpi=DPI)
plt.show()

## 6. Cierre y resumen

**Control:** las salidas del `03` y del `04` no cambiaron. Todo lo que sigue es exploratorio y no modifica el resultado de la regla.

In [ ]:
assert huella_archivos() == HUELLA_INICIAL, 'Cambiaron las salidas del 03 o del 04'   # control 8
print(f'Control de salidas OK · {(time.time() - T_INICIO) / 60:.1f} min en total')
if segundos:
    print('Segundos por modelo:', {c: round(float(np.mean(v)), 1) for c, v in segundos.items()})
print()
print('EXPLORATORIO: no cambia el resultado de la regla pre-registrada.')
print()
print('Ensamble contra A (bootstrap de clientes):')
for r in ensamble_vs_a.itertuples():
    print(f'  {r.foto_mes} · {r.config}: {coma(r.d_M, 2)} M (IC 95 %: {coma(r.ic95_inf_M, 1)} a {coma(r.ic95_sup_M, 1)})')
print()
print('Segundas diferencias (por semilla, p descriptivo):')
for r in d2_comparacion.itertuples():
    print(f'  {r.foto_mes} · {r.config} − {r.contra}: {coma(r.d_media_M, 2)} M (IC 95 %: {coma(r.ic95_inf_M, 1)} a '
          f'{coma(r.ic95_sup_M, 1)}), {r.semillas_a_favor} de {len(SEMILLAS_D2)} a favor, p {coma(r.p_descriptivo, 3)}')